# Pretraining Hyformer on ZINC250k

This notebook pretrains Hyformer from scratch with the causal (LM) and masked (MLM) language modelling objectives on the whole ZINC250k dataset and samples a few molecules from the result.

In [1]:
# Imports and repo root
%load_ext autoreload
%autoreload 2

from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from rdkit import Chem
from rdkit.Chem import Draw

from joint_improvement.hyformer import Hyformer, HyformerConfig
from joint_improvement.tokenizers import SMILESTokenizer
from joint_improvement.trainers import MultiTaskTrainer
from joint_improvement.trainers.multitask import TrainerConfig
from joint_improvement.utils.chemistry import calculate_validity
from joint_improvement.utils.data.collators import LMCollator, MLMCollator
from joint_improvement.utils.data.data_io import load_npz_data, save_npz_data
from joint_improvement.utils.data.dataloader import SequenceDataLoader
from joint_improvement.utils.data.dataset import SequenceDataset

REPO_ROOT = Path.cwd().parent  # notebook is run from notebooks/

/Users/adamizdebskihm/mamba/envs/joint-improvement/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## ZINC250k dataset

ZINC250k is a random subset of ~250,000 drug-like molecules from the ZINC database. It was popularized by the Chemical VAE (Gómez-Bombarelli et al., 2018) and has since become a standard benchmark in molecular machine learning.

First, download ZINC250k to `data/zinc250k/raw/`.

Then split the whole dataset (SMILES only, no labels) into a training set (99%) and a held-out validation set (1%) and save them to `data/zinc250k/unsupervised/`, where the dataset configs in `configs/datasets/zinc250k/unsupervised/` expect them. The validation loss selects the best checkpoint.

In [2]:
# Download the ZINC250k CSV (skipped if already present)
ZINC_URL = (
    "https://raw.githubusercontent.com/aspuru-guzik-group/chemical_vae/master/"
    "models/zinc_properties/250k_rndm_zinc_drugs_clean_3.csv"
)
raw_path = REPO_ROOT / "data/zinc250k/raw/250k_rndm_zinc_drugs_clean_3.csv"
raw_path.parent.mkdir(parents=True, exist_ok=True)

if not raw_path.exists():
    urlretrieve(ZINC_URL, raw_path)
raw_path

PosixPath('/Users/adamizdebskihm/Code/joint-improvement/data/zinc250k/raw/250k_rndm_zinc_drugs_clean_3.csv')

In [3]:
# Split the whole ZINC250k into train (99%) and validation (1%) sets of SMILES
smiles_all = pd.read_csv(raw_path)["smiles"].str.strip().to_numpy()  # SMILES in this CSV carry a trailing newline

VAL_FRACTION = 0.01
rng = np.random.default_rng(0)
out_dir = REPO_ROOT / "data/zinc250k/unsupervised"
out_dir.mkdir(parents=True, exist_ok=True)  # save_npz_data does not create the directory

perm = rng.permutation(len(smiles_all))
n_val = int(len(smiles_all) * VAL_FRACTION)
splits = {"train": perm[n_val:], "val": perm[:n_val]}

for name, idx in splits.items():
    # explicit str dtype: an object array would need pickle to load
    save_npz_data(out_dir / f"{name}.npz", {"sequence": np.asarray(smiles_all[idx], dtype=str)})
    print(f"{name}: n={len(idx)}")

train: n=246961
val: n=2494


## Hyformer: pretrain with LM and MLM objectives

Initialized from scratch and trained on the whole ZINC250k.

In [11]:
# Set the device and define the constants of this section
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

TOKENIZER_CONFIG_PATH = REPO_ROOT / "configs/tokenizers/smiles/zinc250k/tokenizer_config.json"
VOCAB_PATH = REPO_ROOT / "data/vocabularies/zinc250k.txt"
MODEL_CONFIG_PATH = REPO_ROOT / "configs/hyformer/base_8m.json"

In [12]:
# Quick installation check: repo root and torch/device
assert (REPO_ROOT / "configs").is_dir(), f"REPO_ROOT looks wrong: {REPO_ROOT}"
print(f"torch {torch.__version__}, device: {device}")

torch 2.14.1, device: mps


In [13]:
# Load the tokenizer, initialize the base model from scratch and print its size
tokenizer = SMILESTokenizer.from_pretrained(TOKENIZER_CONFIG_PATH, vocab_filepath=VOCAB_PATH)

config = HyformerConfig.from_pretrained(MODEL_CONFIG_PATH)
model = Hyformer.from_config(config).to(device).eval()  # from_config adds generate() (see generator_type)

n_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters: {n_params:,} ({n_params / 1e6:.2f}M)")

Number of parameters: 6,838,272 (6.84M)


In [ ]:
# Pretrain Hyformer (causal LM + masked LM) on the whole ZINC250k; the validation loss selects the best checkpoint
# pretrain.json needs CUDA (bf16, torch.compile); pretrain_mps.json is the float32 variant for Apple silicon / CPU
TRAINER_CONFIG_PATH = REPO_ROOT / f"configs/trainers/{'pretrain' if device == 'cuda' else 'pretrain_mps'}.json"
PRETRAIN_DIR = REPO_ROOT / "results/zinc250k/pretrain"  # the best checkpoint (lowest validation loss) is saved here

trainer_config = TrainerConfig.from_pretrained(TRAINER_CONFIG_PATH)
COLLATORS = {"lm": LMCollator, "mlm": MLMCollator}


def make_loader(name: str, task: str, shuffle: bool) -> SequenceDataLoader:
    data = load_npz_data(out_dir / f"{name}.npz", sequence_key="sequence")
    return SequenceDataLoader(
        SequenceDataset(data["sequence"]),
        COLLATORS[task](tokenizer, max_length=config.max_seq_len),
        batch_size=trainer_config.batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=False,
    )


trainer = MultiTaskTrainer(trainer_config, model, device=device, out_dir=PRETRAIN_DIR)
trainer.train(
    train_loaders={task: make_loader("train", task, shuffle=True) for task in trainer_config.tasks},
    val_loaders={task: make_loader("val", task, shuffle=False) for task in trainer_config.tasks},
)  # the best weights are reloaded into `model` at the end

print(f"Best weighted validation loss: {trainer.state.best_val_loss:.4f} (epoch {trainer.state.best_epoch})")

2026-10-06 08:51:32.095 | INFO     | joint_improvement.trainers.device_optimizations:setup_device_optimizations:49 - Training with dtype: torch.float32


/Users/adamizdebskihm/Code/joint-improvement/src/joint_improvement/trainers/device_optimizations.py:54: UserWarning: In MPS autocast, but the target dtype is not supported. Disabling autocast.
MPS Autocast only supports dtypes of torch.bfloat16, torch.float16 currently.
  compile_mode = "default"
2026-10-06 08:52:37.013 | INFO     | joint_improvement.trainers.multitask:train:511 - Step 100: task lm, loss 1.9821, grad_norm 4.5882, lr 0.000062, tokens/s 25274
2026-10-06 08:53:42.527 | INFO     | joint_improvement.trainers.multitask:train:511 - Step 200: task lm, loss 1.3907, grad_norm 4.7244, lr 0.000124, tokens/s 25009
2026-10-06 08:54:49.236 | INFO     | joint_improvement.trainers.multitask:train:511 - Step 300: task lm, loss 1.1335, grad_norm 5.9738, lr 0.000187, tokens/s 24560
2026-10-06 08:56:00.834 | INFO     | joint_improvement.trainers.multitask:train:511 - Step 400: task lm, loss 1.0638, grad_norm 11.1832, lr 0.000249, tokens/s 22884
2026-10-06 08:57:13.770 | INFO     | joint_im

In [11]:
# Generate SMILES: start every sequence from [<lm> task token, <bos>] and sample token by token
# Sampling hyperparameters
N_GENERATED = 10  # number of sequences sampled in parallel
MAX_SEQUENCE_LENGTH = config.max_seq_len  # prefix + generated tokens
TEMPERATURE = 1.0  # <1 = more conservative samples, >1 = more diverse
TOP_K = None  # sample only from the k most likely tokens (None = off)
TOP_P = None  # nucleus sampling: smallest token set with cumulative probability >= TOP_P (None = off)
SEED = 0  # seed of the sampling random number generator

model.eval()
generator = torch.Generator(device=device).manual_seed(SEED)
prefix_ids = torch.tensor([[tokenizer.task_token_ids["lm"], tokenizer.bos_token_id]] * N_GENERATED, device=device)
generated_ids = model.generate(
    prefix_input_ids=prefix_ids,
    max_sequence_length=MAX_SEQUENCE_LENGTH - prefix_ids.shape[1],
    eos_token_id=tokenizer.eos_token_id,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    top_p=TOP_P,
    generator=generator,
)

generated_smiles = []
for ids in generated_ids[:, prefix_ids.shape[1] :].tolist():
    if tokenizer.eos_token_id in ids:  # sequences that finished early keep sampling, so cut at the first <eos>
        ids = ids[: ids.index(tokenizer.eos_token_id)]
    generated_smiles.append(tokenizer.decode(ids))

for smiles in generated_smiles:
    is_valid = calculate_validity(smiles)
    print(f"valid={is_valid}  {smiles}")
    if is_valid:  # invalid SMILES cannot be drawn
        display(Draw.MolToImage(Chem.MolFromSmiles(smiles), size=(250, 200)))

valid=False  cIcc8[P@@H](NNccc[n+]c
valid=False  CO[C@@]C(C)C(=[C@]N(=O)C=O)CCC1(=7ccc3C[n+])C1
valid=False  CCC(Sc1[C@@](=O)cc[P@]c2-[S-]c3=CN(=O)CC1Nc3[n+]1
valid=False  CNc1C(=O)O)NcO[NH+]C
valid=False  OCc[NH2+]P4CC=[N-]C1c)-C
valid=False  C7c1c1/C(=OCNncc[o+]Oc3\cc(CN[C@@H](N2)c(CC)c2c[CH2-]C)cc2)Cc[S@]
valid=False  CNFc1C2C3c[O-]cc3[P@@H]n2C
valid=False  COcIN[P@][O-]CC(CN(C)=Occ(=O)Nccc[O-]C1
valid=False  C\(=O)
valid=False  C[O-]17[C@H](=oC(C[NH+]Cnc1cC[C@H]NC[S@@]CC(C(N)C[n+])CCs1
